# 03 - Silver Layer Validation

Validate Silver layer transformations — deduplication, cleansing, and AML flagging.

In [ ]:
import sys
sys.path.insert(0, '/opt/spark-jobs')
from utils.spark_session import create_spark_session
from utils.delta_helpers import read_delta
from pyspark.sql import functions as F

spark = create_spark_session(app_name='SilverValidation')

In [ ]:
# Compare Bronze vs Silver record counts
bronze_txn = read_delta(spark, 's3a://bronze/transactions')
silver_txn = read_delta(spark, 's3a://silver/transactions_cleansed')

print(f'Bronze transactions: {bronze_txn.count()}')
print(f'Silver transactions: {silver_txn.count()}')
print(f'Records filtered:   {bronze_txn.count() - silver_txn.count()}')

In [ ]:
# Verify no duplicate transaction_ids in Silver
dupes = silver_txn.groupBy('transaction_id').count().filter(F.col('count') > 1)
print(f'Duplicate transaction_ids: {dupes.count()}')

In [ ]:
# Verify all amounts are positive
invalid = silver_txn.filter(F.col('amount') <= 0).count()
print(f'Invalid amounts (<=0): {invalid}')

In [ ]:
# AML flag distribution
aml_flagged = silver_txn.filter(F.col('_aml_flag') == True).count()
total = silver_txn.count()
print(f'AML Flagged: {aml_flagged} / {total} ({aml_flagged/total*100:.1f}%)')

In [ ]:
spark.stop()